# UnetSemanticSegmentationExample notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TXSTCODEPLAYGROUND/CodeCS4337Fall2026/blob/main/UnetSemanticSegmentationExample/unet_semantic_segmentation_example_notebook.ipynb)

This notebook teaches **semantic segmentation** with a **U-Net**: for every pixel of a colonoscopy image, the network decides whether it belongs to a **polyp** (a small growth on the wall of the colon, which doctors look for because some turn into cancer) or to the background. The data is **Kvasir-SEG**: 1,000 colonoscopy images, each with a hand-drawn mask of its polyp. What the project does and how its code is organized is in its [README](README.md).

The notebook follows the path of a segmentation project from start to finish:

**Dataset → Exploration → Tensors → DataLoader → U-Net → Inspection → Loss → Training → Evaluation → Predictions → Comparison → Fine-tuning → Extensions**

You will build each piece in readable PyTorch, print the shape of every tensor along the way, train a U-Net with a plain PyTorch loop, then run three full experiments with the project (PyTorch Lightning + Weights & Biases) and compare them:

| Config | Experiment | Model | Loss | Optimizer and schedule | Augmentation |
| --- | --- | --- | --- | --- | --- |
| `config01.json` | Baseline | U-Net from scratch (7.8 million parameters) | BCE (binary cross-entropy) | Adam, constant learning rate | Resize + horizontal flip |
| `config02.json` | Improved training | The same U-Net | BCE + Dice | AdamW, cosine schedule | Strong: crops, flips, rotations, colors |
| `config03.json` | Fine-tuning | U-Net with an ImageNet-pretrained ResNet-34 encoder (24.4 million) | BCE + Dice | AdamW, smaller learning rate for the encoder, encoder frozen for 3 epochs, cosine schedule | Strong |

### Choose your route

| Route | Sections to run | Then run the project with |
| --- | --- | --- |
| **Colab, from notebook cells** | 1 (setup), then everything from 2 | Option A1 (`!python -m ...`) or Option B (`main(...)`) in section 11 |
| **Colab, from Colab's terminal** (Colab Pro) | 1 (setup) in the notebook first | Option A2: the same commands in the terminal |
| **Locally, from this notebook** | Skip 1, start at [**2. Common setup**](#2.-Common-setup) | Option A1 or B, with the `.venv` as the notebook's kernel |
| **Locally, from your terminal** | None for training; the notebook for the explanations and plots | `python -m UnetSemanticSegmentationExample --config config01.json` from the repo root, with the `.venv` activated |

In Colab, the setup cells are needed even when you train in the terminal: Google Drive can only be mounted, and Colab Secrets only read, from a notebook cell. Locally, set up the repo once first (`.venv`, requirements, and `.env`, see the main README), and open this notebook from the project folder.

**Use a GPU** in Colab: *Runtime → Change runtime type → GPU*. Sections 3 to 10 also run on a CPU (the training loop of section 10 then trains on fewer batches), but the three experiments of section 11 need a GPU (up to 100 epochs each, about 10 to 20 minutes on a T4; config03 usually stops earlier).

### Sections

| Part | Sections |
| --- | --- |
| Setup | 1. Setup (Colab) · 2. Common setup |
| Data | 3. Segmentation vs. classification · 4. Explore Kvasir-SEG · 5. From files to tensors: Dataset and DataLoader |
| Loss | 6. Binary cross-entropy from first principles |
| Model | 7. The U-Net, block by block · 8. Reading a torchinfo summary · 9. Seeing the graph: Netron and torchview |
| Training | 10. A training loop in plain PyTorch · 11. Three experiments with the project |
| Fine-tuning | 12. Inside the fine-tuned model |
| Evaluation | 13. Learning curves · 14. Predictions side by side · 15. Numbers side by side |
| Reuse | 16. Load a trained model, keep training |
| Extensions | 17. Try it yourself: other segmentation datasets · 18. Experiment ideas · Pro tips |

## 1. Setup

> **Running locally?** Skip this section: these cells only work in Colab. Jump to [**2. Common setup**](#2.-Common-setup).

Run these cells once per session (and again after the runtime resets): mount Drive, set up the repo, change directory, and load API keys. The setup cell is safe to re-run, e.g. to get new projects with `git pull`; run the API-key cell again after it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
set -e

cd /content
if [ -d CodeCS4337Fall2026/.git ]; then
  git -C CodeCS4337Fall2026 pull --ff-only
else
  git clone https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026.git
fi
cd CodeCS4337Fall2026

# Colab settings: data on the local disk, results in Google Drive.
cp .envcolab .env

# Install only packages Colab does not already have (versions are not pinned).
grep -vE '^\s*(#|$)' requirements.txt \
  | sed -E 's/[<>=!~;[ ].*//' \
  | while read -r pkg; do
      if pip show "$pkg" > /dev/null 2>&1; then
        echo "skip $pkg (already installed)"
      else
        pip install -q "$pkg" && echo "installed $pkg"
      fi
    done

In [ ]:
# The bash cell above cannot change the notebook's working directory, so do it here.
%cd /content/CodeCS4337Fall2026

### Load API keys (optional)

This project logs every run to Weights & Biases (W&B), which needs `WANDB_API_KEY`. Without it, runs are logged to W&B offline and can be uploaded later with the `wandb sync` command printed at the end of the run.

1. Add the keys as Colab **Secrets**: key icon in the left sidebar → *Add new secret*, named `WANDB_API_KEY`. Never type keys into a notebook cell or into `.envcolab`, which is public on GitHub.
2. Run the cell below **after the setup cells and before running the project**, whether you run it from this notebook (Option A1 or B) or from Colab's terminal (A2). Run it again whenever you re-run the setup cell.

The cell loads `.env`, then looks up every key that is still missing in your Secrets (Colab may ask you to *Grant access*). Keys it finds are added to this notebook's environment and saved to `/content/CodeCS4337Fall2026/.env`, so terminal runs find them too. That file is not in git and is deleted when the runtime resets. Values are never printed.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google.colab import userdata

ENV_FILE = Path("/content/CodeCS4337Fall2026/.env")
KEYS = ["WANDB_API_KEY"]

if load_dotenv(ENV_FILE):
    print(f"Loaded settings from {ENV_FILE}")
else:
    print(f"No settings in {ENV_FILE}: run the setup cells first")

for key in KEYS:
    if os.getenv(key):
        print(f"{key}: found in .env")
        continue
    try:
        os.environ[key] = userdata.get(key)
    except Exception:  # noqa: BLE001 (Secret not added or access not granted)
        print(f"{key}: not found, W&B will log offline")
        continue
    with ENV_FILE.open("a") as env_file:
        env_file.write(f"\n{key}={os.environ[key]}\n")
    print(f"{key}: loaded from Colab Secrets and saved to .env")

## 2. Common setup

**Run this cell first, in Colab and locally.** It finds the repo folder and makes the project importable, reads the data and output folders from `.env`, sets the random seeds (so every run of this notebook shows the same images and gets the same numbers), and picks the GPU when there is one.

In [ ]:
import os
import sys
from pathlib import Path

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import torch
from dotenv import load_dotenv
from torch.nn import functional as F

# Colab: the clone made by the setup cell. Locally: the parent of this notebook's project folder.
IN_COLAB = "google.colab" in sys.modules
REPO = Path("/content/CodeCS4337Fall2026") if IN_COLAB else Path.cwd().parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# Colab: the copy of .envcolab made by the setup cell. Locally: your own .env.
load_dotenv(REPO / ".env")
DATA_DIR = REPO / os.getenv("DATA_DIR", "data")
OUTPUT_DIR = REPO / os.getenv("OUTPUT_DIR", "runs")

SEED = 42
L.seed_everything(SEED)  # seeds Python's random, NumPy, and PyTorch (CPU and GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Repo folder: {REPO}")
print(f"Data folder: {DATA_DIR}")
print(f"Runs folder: {OUTPUT_DIR / 'UnetSemanticSegmentationExample'}")
print(
    f"Device:      {device}"
    + (f" ({torch.cuda.get_device_name()})" if device.type == "cuda" else "")
)

## 3. Segmentation vs. classification

An image **classifier** answers one question per image: *what is in this image?* A **semantic segmentation** network answers the same question **for every pixel**: *which class does this pixel belong to?* Its output is an image of the same height and width as the input, a **mask**.

Here there are two classes, background (0) and polyp (1): **binary segmentation**. The network outputs one number per pixel, a **logit** $z$ (any real number). The **sigmoid** $\sigma(z) = 1 / (1 + e^{-z})$ turns it into a probability between 0 and 1, and a **threshold** (0.5) turns the probability into the predicted mask:

```text
 image             U-Net              logits            sigmoid           probability map     >= 0.5          predicted mask
[3, 256, 256] ───────────────────► [1, 256, 256] ───────────────────► [1, 256, 256] ───────────────────► [1, 256, 256]
 float, normalized                   any real number                   between 0 and 1                   0 or 1
```

What changes compared with classification, for a batch of $B$ images of $H \times W$ pixels:

| | Image classification | Binary segmentation (this project) | Multiclass segmentation ($C$ classes) |
| --- | --- | --- | --- |
| Input `X` | `[B, 3, H, W]` | `[B, 3, H, W]` | `[B, 3, H, W]` |
| Target `y` | `[B]`, one class index per image | `[B, 1, H, W]`, 0.0 or 1.0 per pixel | `[B, H, W]`, one class index per pixel (`int64`) |
| Network output | `[B, C]` logits | `[B, 1, H, W]` logits | `[B, C, H, W]` logits |
| Probability | softmax over the $C$ classes | sigmoid of each logit | softmax over the $C$ channels of each pixel |
| Loss | cross-entropy, averaged over the images | binary cross-entropy, averaged over **all pixels** of all images | cross-entropy, averaged over all pixels |
| Main metrics | accuracy, F1 | Dice, IoU (overlap of the masks) | mean IoU over the classes |

The network itself must keep the image's resolution: it shrinks the image to understand *what* is there (like a classifier), then grows it back to say *where*. That is the U-Net of section 7.

## 4. Explore Kvasir-SEG

[Kvasir-SEG](https://datasets.simula.no/kvasir-seg/) (Jha et al., 2020) is 1,000 colonoscopy images from Vestre Viken hospitals in Norway, each with a mask drawn by a doctor and checked by an experienced gastroenterologist. It may be used for research and education; cite Jha et al., *Kvasir-SEG: A Segmented Polyp Dataset*, MMM 2020.

The cell below downloads it once (44 MB) to `<DATA_DIR>/kvasir-seg/` and lists the files. Each image in `images/` has a mask with the **same file name** in `masks/`.

In [ ]:
from UnetSemanticSegmentationExample.dataloaders import prepare_kvasir

dataset_dir = prepare_kvasir(DATA_DIR)
image_files = sorted((dataset_dir / "images").glob("*.jpg"))
mask_files = sorted((dataset_dir / "masks").glob("*.jpg"))

print(f"Dataset folder: {dataset_dir}")
print(f"Images: {len(image_files)}, masks: {len(mask_files)}")
print(
    f"Same file names: {[p.name for p in image_files] == [p.name for p in mask_files]}"
)
print(f"First files: {[p.name for p in image_files[:3]]}")

### Image sizes

The images come from different endoscopes and have **many different sizes**. A network trains on batches, and a batch is one tensor, so every image will be resized to the same size (256 x 256 here). `Image.open` only reads the file header, so this is fast.

In [ ]:
from collections import Counter

from PIL import Image

sizes = Counter(Image.open(p).size for p in image_files)  # (width, height)
widths = np.array([w for (w, h), n in sizes.items() for _ in range(n)])
heights = np.array([h for (w, h), n in sizes.items() for _ in range(n)])

print(f"{len(sizes)} different sizes (width x height)")
print(
    f"Smallest: {min(sizes, key=lambda s: s[0] * s[1])}, largest: {max(sizes, key=lambda s: s[0] * s[1])}"
)
print(
    "Most common:",
    ", ".join(f"{w}x{h} ({n} images)" for (w, h), n in sizes.most_common(4)),
)

plt.figure(figsize=(5, 4))
plt.scatter(widths, heights, s=8, alpha=0.4)
plt.xlabel("width (pixels)")
plt.ylabel("height (pixels)")
plt.title("Kvasir-SEG image sizes")
plt.show()

### One image and its mask, as stored on disk

An image file is decoded into a NumPy array of shape `(H, W, 3)` (height, width, color channels) with values from 0 to 255 (`uint8`).

The masks are stored as **JPEG** images too, with 3 identical color channels. JPEG compression is lossy: the mask is not exactly black (0) and white (255) but contains values like 3 or 251. The histogram below shows them, in two narrow groups near 0 and near 255. Comparing with `== 255` would lose polyp pixels; the project turns each mask into exactly two values by thresholding at 128 (`load_binary_mask`): 1 for polyp, 0 for background, and keeps one channel.

In [ ]:
from UnetSemanticSegmentationExample.dataloaders import MASK_THRESHOLD, load_binary_mask

image = np.array(Image.open(image_files[0]))
raw_mask = np.array(Image.open(mask_files[0]))
mask = load_binary_mask(mask_files[0])

print(
    f"image:    shape {image.shape}, dtype {image.dtype}, values {image.min()} to {image.max()}"
)
print(
    f"raw mask: shape {raw_mask.shape}, dtype {raw_mask.dtype}, {len(np.unique(raw_mask))} different values"
)
print(
    f"binary mask (>= {MASK_THRESHOLD}): shape {mask.shape}, dtype {mask.dtype}, values {np.unique(mask)}"
)
print(f"Raw mask values: {np.unique(raw_mask).tolist()}")
print(
    f"Pixels that are not exactly 0 or 255: {((raw_mask != 0) & (raw_mask != 255)).mean():.2%}"
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
axes[0].imshow(image)
axes[0].set_title("image")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("binary mask (white = polyp)")
axes[2].hist(raw_mask.ravel(), bins=64, log=True)
axes[2].axvline(
    MASK_THRESHOLD, color="red", linestyle="--", label=f"threshold {MASK_THRESHOLD}"
)
axes[2].set_title("raw mask values (log scale)")
axes[2].legend()
for ax in axes[:2]:
    ax.axis("off")
plt.show()

### How much of each image is polyp?

The share of polyp pixels (the **foreground**) in each mask decides how hard the problem is. When polyps are small, most pixels are background, and a network that always says "background" is already right for most pixels. This **class imbalance** is why pixel accuracy is a poor metric and why section 6 adds the Dice loss.

In [ ]:
foreground = np.array([load_binary_mask(p).mean() for p in mask_files])

print(
    f"Polyp share per image: mean {foreground.mean():.1%}, median {np.median(foreground):.1%}, "
    f"min {foreground.min():.1%}, max {foreground.max():.1%}"
)
print(
    f"Images where the polyp covers less than 5% of the image: {(foreground < 0.05).sum()}"
)
print(
    f"Over the whole dataset, {1 - foreground.mean():.1%} of the pixels are background"
)

plt.figure(figsize=(6, 3.5))
plt.hist(foreground * 100, bins=40)
plt.xlabel("polyp pixels (% of the image)")
plt.ylabel("number of images")
plt.title("Foreground share per mask")
plt.show()

### Examples, from the smallest polyp to the largest

Each row: the image, its mask, and the mask drawn over the image (an **overlay**: polyp pixels tinted green, with the outline). Overlays are the best way to check that masks and images line up.

In [ ]:
def overlay(
    image: np.ndarray, mask: np.ndarray, color=(0, 255, 0), alpha: float = 0.4
) -> np.ndarray:
    """Tint the mask's pixels in ``color`` on top of an (H, W, 3) uint8 image."""
    tinted = image.astype(np.float32)
    tinted[mask > 0] = (1 - alpha) * tinted[mask > 0] + alpha * np.array(color)
    return tinted.astype(np.uint8)


order = np.argsort(foreground)
examples = [order[int(q * (len(order) - 1))] for q in (0.0, 0.25, 0.5, 0.75, 1.0)]
fig, axes = plt.subplots(
    len(examples), 3, figsize=(11, 3.2 * len(examples)), layout="constrained"
)
for row, i in zip(axes, examples):
    img = np.array(Image.open(image_files[i]).convert("RGB"))
    msk = load_binary_mask(mask_files[i])
    row[0].imshow(img)
    row[0].set_title(
        f"{image_files[i].name}  {img.shape[1]}x{img.shape[0]}", fontsize=9
    )
    row[1].imshow(msk, cmap="gray")
    row[1].set_title(f"mask: {foreground[i]:.1%} polyp", fontsize=9)
    row[2].imshow(overlay(img, msk))
    row[2].contour(msk, levels=[0.5], colors="lime", linewidths=1)
    row[2].set_title("overlay", fontsize=9)
    for ax in row:
        ax.axis("off")
plt.show()

**What to notice:**

- Polyps vary a lot in size, shape, color, and position; some are flat and hard to see even for a person.
- Many images have a **black border** and some have a **green box** in the corner (an overlay of the endoscope's position display). The network must learn to ignore them.
- Specular highlights (white reflections of the light) appear on polyps and on the normal colon wall alike.

## 5. From files to tensors: Dataset and DataLoader

A network needs **tensors** of a fixed shape, normalized like the data it was designed for. The project's `KvasirSegDataset` (in `dataloaders/kvasir_seg.py`) does this for one image and its mask at a time:

1. **Load** the image as RGB and the mask, binarized at 128.
2. **Wrap** them as `tv_tensors.Image` `(3, H, W)` and `tv_tensors.Mask` `(1, H, W)`. These tell torchvision's `transforms.v2` which is which, so that one call transforms **both with the same random parameters**: a flip flips the image and the mask together. Masks are resized with *nearest-neighbor* interpolation, so they stay 0 and 1 (interpolating between 0 and 1 would create values like 0.5 that are not a class).
3. **Resize** to 256 x 256 (and augment, for training images).
4. **Convert** the image to `float32` between 0 and 1, then **normalize** it with the ImageNet mean and standard deviation of each color channel (the statistics the pretrained ResNet-34 of config03 was trained with).
5. Return the mask as `float32` 0.0 / 1.0, the type the loss function expects.

The cell below runs these steps by hand on one image and prints the shape, type, and range after each step.

In [ ]:
from torchvision import tv_tensors

from UnetSemanticSegmentationExample.dataloaders import (
    IMAGENET_MEAN,
    IMAGENET_STD,
    build_transform,
)


def describe(name: str, t) -> None:
    """Print the shape, dtype, and value range of an array or tensor."""
    t = torch.as_tensor(np.asarray(t)) if not isinstance(t, torch.Tensor) else t
    print(
        f"{name:<34} shape {tuple(t.shape)!s:<16} dtype {t.dtype!s:<14} "
        f"values {t.min().item():.3f} to {t.max().item():.3f}"
    )


name = image_files[0].name
pil_image = Image.open(dataset_dir / "images" / name).convert("RGB")
array = np.array(pil_image)
describe("1. image as NumPy (H, W, C)", array)
image_t = tv_tensors.Image(torch.from_numpy(array).permute(2, 0, 1))
describe("2. tv_tensors.Image (C, H, W)", image_t)
mask_t = tv_tensors.Mask(
    torch.from_numpy(load_binary_mask(dataset_dir / "masks" / name))[None]
)
describe("2. tv_tensors.Mask (1, H, W)", mask_t)

eval_transform = build_transform(image_size=256, augmentation="none", train=False)
print(eval_transform)
image_out, mask_out = eval_transform(image_t, mask_t)
describe("3-4. image resized and normalized", image_out)
describe("3. mask resized (nearest)", mask_out)
describe("5. mask as float, the loss target", mask_out.float())
print(
    f"Mask values after resizing: {torch.unique(mask_out).tolist()}  (still only 0 and 1)"
)

### Augmentation keeps image and mask aligned

`config02` and `config03` use **strong augmentation**: a random crop of 50 to 100% of the image resized to 256 x 256, horizontal and vertical flips, rotations up to 15°, and small brightness, contrast, saturation, and hue changes (color changes are applied to the image only). Every epoch the network sees a different version of each training image, which makes it harder to memorize the 800 images (**overfitting**).

Below, the same image goes through the strong training transform four times. The green outline is the transformed mask: it follows the polyp in every version. To show a normalized image, the normalization is undone (`denormalize`).

In [ ]:
MEAN = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
STD = torch.tensor(IMAGENET_STD).view(3, 1, 1)


def denormalize(image: torch.Tensor) -> np.ndarray:
    """Normalized (3, H, W) tensor to an (H, W, 3) uint8 image for display."""
    image = image.detach().cpu() * STD + MEAN
    return (image.clamp(0, 1) * 255).byte().permute(1, 2, 0).numpy()


strong = build_transform(image_size=256, augmentation="strong", train=True)
torch.manual_seed(SEED)
fig, axes = plt.subplots(1, 5, figsize=(18, 4), layout="constrained")
axes[0].imshow(denormalize(image_out))
axes[0].contour(mask_out[0], levels=[0.5], colors="lime")
axes[0].set_title("evaluation transform")
for k, ax in enumerate(axes[1:], start=1):
    aug_image, aug_mask = strong(image_t, mask_t)
    ax.imshow(denormalize(aug_image))
    ax.contour(aug_mask[0], levels=[0.5], colors="lime")
    ax.set_title(f"strong augmentation #{k}")
for ax in axes:
    ax.axis("off")
plt.show()

### The DataModule and its DataLoaders

`KvasirSegDataModule` splits the 1,000 images, with the seed, into **800 training, 100 validation, and 100 test images**, creates one `KvasirSegDataset` per split, and wraps each in a PyTorch `DataLoader`, which stacks `batch_size` samples into one tensor and loads them in parallel worker processes (`num_workers`). Only the training loader shuffles, and only training images are augmented.

In [ ]:
from UnetSemanticSegmentationExample.dataloaders import KvasirSegDataModule

data = KvasirSegDataModule(
    str(DATA_DIR), image_size=256, batch_size=8, num_workers=2, augmentation="minimal"
)
data.prepare_data()
data.setup("fit")
print(
    f"Split sizes: train {len(data.train_set)}, validation {len(data.val_set)}, test {len(data.test_set)}"
)

images, masks = next(iter(data.train_dataloader()))
describe("images batch [B, C, H, W]", images)
describe("masks batch  [B, 1, H, W]", masks)
print(f"Mask values in the batch: {torch.unique(masks).tolist()}")
print("Polyp share of each image in the batch:", [f"{m.mean():.1%}" for m in masks])

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8), layout="constrained")
for ax, image, mask in zip(axes.flat, images, masks):
    ax.imshow(overlay(denormalize(image), mask[0].numpy()))
    ax.contour(mask[0], levels=[0.5], colors="lime", linewidths=1)
    ax.set_title(f"image {tuple(image.shape)}, mask {tuple(mask.shape)}", fontsize=9)
    ax.axis("off")
fig.suptitle("One training batch: images with their masks (green)")
plt.show()

## 6. Binary cross-entropy from first principles

The network outputs a logit $z$ for each pixel; $p = \sigma(z)$ is its predicted probability of polyp, and $y \in \{0, 1\}$ is the true label. The **binary cross-entropy** (BCE) of one pixel is

$$\text{BCE}(p, y) = -\big[\, y \log p + (1 - y) \log (1 - p) \,\big]$$

Only one of the two terms is active: for a polyp pixel ($y = 1$) the loss is $-\log p$, for a background pixel ($y = 0$) it is $-\log(1 - p)$. It is 0 when the prediction is certain and right, and grows without limit as a confident prediction becomes wrong.

### One pixel by hand

A polyp pixel ($y = 1$) with logit $z = 1.5$: $p = \sigma(1.5) = 1 / (1 + e^{-1.5}) \approx 0.8176$, so the loss is $-\log 0.8176 \approx 0.2014$. If the same pixel were background ($y = 0$), the loss would be $-\log(1 - 0.8176) \approx 1.7014$: the network was 82% sure of the wrong answer.

In [ ]:
z = torch.tensor(1.5)
p = 1 / (1 + torch.exp(-z))
print(f"z = {z:.1f}  ->  p = sigmoid(z) = {p:.4f}")

for y in (1.0, 0.0):
    manual = -(y * torch.log(p) + (1 - y) * torch.log(1 - p))
    pytorch = F.binary_cross_entropy_with_logits(z, torch.tensor(y))
    print(
        f"y = {y:.0f}: by hand {manual:.4f}, F.binary_cross_entropy_with_logits {pytorch:.4f}"
    )

### A 2 x 2 "image" by hand

A segmentation loss is the same formula at every pixel, then a **reduction**: by default the **mean over all pixels** (`reduction="mean"`). `reduction="none"` keeps one loss per pixel, and `"sum"` adds them.

In [ ]:
logits = torch.tensor([[2.0, -1.0], [0.5, -3.0]])
target = torch.tensor([[1.0, 0.0], [0.0, 1.0]])

probs = torch.sigmoid(logits)
per_pixel = -(target * torch.log(probs) + (1 - target) * torch.log(1 - probs))
print(" pixel   logit z   p = sigmoid(z)   label y   loss")
for (r, c), zz in np.ndenumerate(logits.numpy()):
    print(
        f" ({r},{c})   {zz:6.2f}      {probs[r, c]:.4f}          {target[r, c]:.0f}     {per_pixel[r, c]:.4f}"
    )
print(f"\nBy hand:   mean {per_pixel.mean():.4f}, sum {per_pixel.sum():.4f}")
print(
    f"PyTorch:   mean {F.binary_cross_entropy_with_logits(logits, target):.4f}, "
    f"sum {F.binary_cross_entropy_with_logits(logits, target, reduction='sum'):.4f}"
)
print(
    "reduction='none':\n",
    F.binary_cross_entropy_with_logits(logits, target, reduction="none"),
)

**Reading the table:** pixel (0,0) is a confident, correct polyp prediction (small loss). Pixel (1,1) is a polyp predicted as background with $p = 0.047$, which dominates the mean. Mistakes made with confidence cost the most.

### With real shapes: `[B, 1, H, W]`

In training, logits and masks are `[B, 1, H, W]` tensors. The loss is computed for each of the $B \times H \times W$ pixels and averaged into **one number**, the value `loss.backward()` differentiates.

In [ ]:
fake_logits = torch.randn(masks.shape)  # what an untrained network might output
per_pixel = F.binary_cross_entropy_with_logits(fake_logits, masks, reduction="none")
print(f"logits {tuple(fake_logits.shape)}, masks {tuple(masks.shape)}")
print(
    f"reduction='none' -> {tuple(per_pixel.shape)}: one loss per pixel, {per_pixel.numel():,} values"
)
print(
    f"reduction='mean' -> {F.binary_cross_entropy_with_logits(fake_logits, masks):.4f} "
    f"(the same as per_pixel.mean() = {per_pixel.mean():.4f})"
)

### Why `BCEWithLogitsLoss` and not `sigmoid` followed by `BCELoss`

Both compute the same formula, but in a different order:

- **sigmoid, then `BCELoss`**: computes $p = \sigma(z)$ first, then $\log p$. For a very negative logit, $\sigma(z)$ is smaller than the smallest `float32` number and **rounds to exactly 0**; $\log 0 = -\infty$. PyTorch clamps the log at $-100$, so the loss is wrong and its **gradient is 0**: the network gets no signal to fix its worst mistakes.
- **`BCEWithLogitsLoss`** (`F.binary_cross_entropy_with_logits`) works on the logit directly, with the algebraically equal but stable form $\max(z, 0) - z\,y + \log\big(1 + e^{-|z|}\big)$, which never takes the log of 0. It is also one fused operation, slightly faster, and the right choice with mixed precision (16-bit), where $\sigma(z)$ rounds to 0 or 1 much sooner.

That is why the U-Net outputs logits, not probabilities, and the sigmoid is applied only for predictions and metrics.

In [ ]:
for z_value in (-5.0, -20.0, -120.0):
    y = torch.tensor([1.0])
    z1 = torch.tensor([z_value], requires_grad=True)
    loss_two_steps = F.binary_cross_entropy(torch.sigmoid(z1), y)
    loss_two_steps.backward()
    z2 = torch.tensor([z_value], requires_grad=True)
    loss_logits = F.binary_cross_entropy_with_logits(z2, y)
    loss_logits.backward()
    print(
        f"z = {z_value:7.1f}, y = 1: sigmoid(z) = {torch.sigmoid(z1).item():.3e} | "
        f"sigmoid + BCELoss: loss {loss_two_steps.item():7.3f}, gradient {z1.grad.item():+.3f} | "
        f"BCEWithLogits: loss {loss_logits.item():7.3f}, gradient {z2.grad.item():+.3f}"
    )

At $z = -120$ the right loss is 120 with gradient $-1$ (push $z$ up); the two-step version reports 100 with gradient 0.

### Class imbalance, and the Dice loss

BCE averages over pixels, so with 85% background, a network that predicts "background, fairly sure" everywhere already gets a low loss and 85% pixel accuracy, while finding **no polyp at all**. The **Dice coefficient** measures the overlap of the predicted and the true polyp only:

$$\text{Dice} = \frac{2\,|P \cap T|}{|P| + |T|} = \frac{2\,TP}{2\,TP + FP + FN}$$

where $P$ is the predicted polyp, $T$ the true one, $TP$ the correctly found polyp pixels, $FP$ background predicted as polyp, and $FN$ polyp predicted as background. Background pixels correctly predicted as background do not appear at all. The **soft Dice loss** $1 - \text{Dice}$ uses the probabilities instead of the 0/1 prediction so that it has a gradient (`dice_loss` in `models/losses.py`). `config02` and `config03` train with **BCE + Dice**: BCE gives a smooth signal at every pixel, Dice keeps the focus on the polyp. **IoU** (intersection over union, $TP / (TP + FP + FN)$) is the other usual overlap metric; it is always lower than Dice for the same prediction.

In [ ]:
from UnetSemanticSegmentationExample.models import dice_loss

all_background = torch.full(
    masks.shape, -3.0
)  # p = 0.047 everywhere: "background, fairly sure"
perfect = torch.where(masks > 0, 5.0, -5.0)  # right everywhere, fairly sure

for name, z in (("always background", all_background), ("perfect", perfect)):
    pred = (torch.sigmoid(z) >= 0.5).float()
    tp, fp, fn = (
        (pred * masks).sum(),
        (pred * (1 - masks)).sum(),
        ((1 - pred) * masks).sum(),
    )
    print(
        f"{name:>17}: pixel accuracy {(pred == masks).float().mean():.3f}, "
        f"BCE {F.binary_cross_entropy_with_logits(z, masks):.3f}, "
        f"Dice {2 * tp / (2 * tp + fp + fn):.3f}, Dice loss {dice_loss(z, masks):.3f}"
    )

## 7. The U-Net, block by block

The U-Net (Ronneberger, Fischer, and Brox, 2015) has the shape of a U:

```text
 input [3, 256, 256]                                                              logits [1, 256, 256]
   │                                                                                   ▲
 encoders[0] DoubleConv ──► [32, 256, 256] ─────────── skip ───────────► concat ─► decoders[3] ─► [32, 256, 256] ─► head (1x1 conv)
   │ max-pool /2                                                          ▲ up x2
 encoders[1] DoubleConv ──► [64, 128, 128] ──────── skip ────────► concat ─► decoders[2] ─► [64, 128, 128]
   │ max-pool /2                                                   ▲ up x2
 encoders[2] DoubleConv ──► [128, 64, 64] ───── skip ─────► concat ─► decoders[1] ─► [128, 64, 64]
   │ max-pool /2                                            ▲ up x2
 encoders[3] DoubleConv ──► [256, 32, 32] ── skip ──► concat ─► decoders[0] ─► [256, 32, 32]
   │ max-pool /2                                     ▲ up x2
 bottleneck  DoubleConv ──► [512, 16, 16] ───────────┘

 left side: ENCODER (contracting path)          bottom: BOTTLENECK          right side: DECODER (expanding path)
```

- **Encoder** (left, going down): each level is a `DoubleConv` (two 3 x 3 convolutions, each with batch normalization and ReLU) followed by a 2 x 2 **max-pooling** that halves the height and width. The channels double at each level (32 → 64 → 128 → 256). Deeper levels see a larger part of the image (a larger **receptive field**) and learn *what* is there, but lose the exact *where*.
- **Bottleneck** (bottom): the smallest, most abstract feature map, 512 channels of 16 x 16.
- **Decoder** (right, going up): each `UpBlock` doubles the height and width with a **transposed convolution** (a learned upsampling) and halves the channels.
- **Skip connections** (horizontal arrows): the decoder **concatenates** the upsampled feature map with the encoder's feature map of the same size. The encoder's maps still hold the fine details (edges, exact positions) lost by pooling; the decoder combines them with the deep "what" information. Without skip connections, the masks come out blurry.
- **Head**: a 1 x 1 convolution turns the 32 channels of each pixel into **one logit**.

Here is the project's code (`models/components/unet.py`): three small classes.

In [ ]:
import inspect

from IPython.display import Code, display

from UnetSemanticSegmentationExample.models.components import unet as unet_module

display(Code(inspect.getsource(unet_module), language="python"))

### Concatenation, the skip connection in one line

`torch.cat([skip, x], dim=1)` stacks two feature maps of the same height and width along **dimension 1, the channels**: 256 channels from the encoder plus 256 from the decoder give 512 channels. Nothing is added or mixed yet; the following `DoubleConv` learns how to combine them.

In [ ]:
skip = torch.randn(1, 256, 32, 32)  # encoders[3]'s output, kept during the way down
upsampled = torch.randn(
    1, 256, 32, 32
)  # the bottleneck, after the transposed convolution
joined = torch.cat([skip, upsampled], dim=1)
print(
    f"skip {tuple(skip.shape)} + upsampled {tuple(upsampled.shape)} -> concatenated {tuple(joined.shape)}"
)
print(
    "Channels 0-255 are the skip, 256-511 the upsampled map:",
    torch.equal(joined[:, :256], skip),
    torch.equal(joined[:, 256:], upsampled),
)

### Shapes through the network

The cell below runs a batch of 2 images through the U-Net **one block at a time**, exactly like `UNet.forward`, and prints the shape after every step. Watch the height and width halve on the way down and double on the way up, and the channels do the opposite. Each skip connection pairs maps of the same size.

In [ ]:
from UnetSemanticSegmentationExample.models import UNet

unet = UNet(in_channels=3, out_channels=1, base_channels=32, depth=4).eval()
x = images[:2]
print(f"{'input':<40} {tuple(x.shape)}")
skips = []
with torch.no_grad():
    for k, encoder in enumerate(unet.encoders):
        if k > 0:
            x = unet.pool(x)
            print(f"{'  max-pool':<40} {tuple(x.shape)}")
        x = encoder(x)
        skips.append(x)
        print(f"{f'encoders[{k}] DoubleConv (kept as skip {k})':<40} {tuple(x.shape)}")
    x = unet.pool(x)
    print(f"{'  max-pool':<40} {tuple(x.shape)}")
    x = unet.bottleneck(x)
    print(f"{'bottleneck DoubleConv':<40} {tuple(x.shape)}")
    for k, (decoder, skip) in enumerate(zip(unet.decoders, reversed(skips))):
        x = decoder.up(x)
        print(f"{f'decoders[{k}].up transposed conv':<40} {tuple(x.shape)}")
        x = torch.cat([skip, x], dim=1)
        print(
            f"{f'  concat with skip {len(skips) - 1 - k} {tuple(skip.shape)}':<40} {tuple(x.shape)}"
        )
        x = decoder.conv(x)
        print(f"{f'decoders[{k}].conv DoubleConv':<40} {tuple(x.shape)}")
    logits = unet.head(x)
    print(f"{'head 1x1 conv: logits':<40} {tuple(logits.shape)}")

    print(
        f"\nSame as unet(images[:2]): {torch.allclose(logits, unet(images[:2]), atol=1e-5)}"
    )
probs = torch.sigmoid(logits)
print(
    f"probabilities {tuple(probs.shape)}, between {probs.min():.3f} and {probs.max():.3f} "
    "(untrained: close to 0.5 everywhere)"
)

## 8. Reading a torchinfo summary

`torchinfo.summary` runs one forward pass and lists every layer with its output shape and number of parameters, like Keras' `model.summary()`.

In [ ]:
from torchinfo import summary

summary(
    UNet(),
    input_size=(1, 3, 256, 256),
    depth=3,
    col_names=("input_size", "output_size", "num_params", "trainable"),
    row_settings=("var_names",),
)

**How to read it:**

- **Rows** are the modules, indented by nesting: `encoders` → `0` (a `DoubleConv`) → `block` (its `Sequential`) → the layers. `depth=3` stops at the third level; use a larger depth to see every `Conv2d` and `BatchNorm2d`.
- **Input / Output Shape**: the shape before and after each module, for a batch of 1. The encoder rows show the spatial size halving, the decoder rows doubling.
- **Param #**: the parameters of that row (including its children). Check a few by hand:
  - The first convolution, 3 → 32 channels with 3 x 3 kernels and no bias (batch normalization follows): $3 \times 32 \times 3 \times 3 = 864$.
  - A `BatchNorm2d(32)` learns a scale and a shift per channel: $2 \times 32 = 64$.
  - `decoders.0.up`, a transposed convolution 512 → 256 with 2 x 2 kernels and a bias: $512 \times 256 \times 2 \times 2 + 256 = 524{,}544$.
  - The `head`, 32 → 1 channel with a 1 x 1 kernel and a bias: $32 + 1 = 33$.
- **Trainable**: `True` when the parameters get gradients. Section 12 shows a frozen encoder, where it reads `False`.
- **Totals** at the bottom: about 7.8 million parameters, most of them in the deepest levels (the bottleneck and `decoders.0`), where the channel counts are largest.
- **Mult-Adds**: the multiply-add operations of one forward pass, a measure of compute. Most of them happen at full resolution: a 3 x 3 convolution on a 256 x 256 map costs 64 times more than the same convolution on 32 x 32.
- **Input size / Forward-backward pass size / Params size (MB)**: the memory for the input, for the activations kept for backpropagation (which grows with the batch size and the image size), and for the weights (4 bytes per parameter in 32-bit). Activations, not weights, decide whether a batch fits on the GPU.

## 9. Seeing the graph: Netron and torchview

[Netron](https://netron.app) draws a network's computation graph from a saved model file. PyTorch models are first exported to **ONNX** (Open Neural Network Exchange), a framework-independent file format that stores the graph and the weights. The cell below exports the U-Net to a temporary file.

In [ ]:
import tempfile

onnx_path = Path(tempfile.gettempdir()) / "unet.onnx"
torch.onnx.export(
    UNet().eval(),
    (torch.randn(1, 3, 256, 256),),
    onnx_path,
    input_names=["image"],
    output_names=["logits"],
    external_data=False,  # weights inside the .onnx file, not in a separate .onnx.data file
    verbose=False,
)
print(
    f"Saved {onnx_path} ({onnx_path.stat().st_size / 1e6:.1f} MB: the graph and the 7.8 million weights)"
)

The cell below starts a small Netron web server in the background and shows it **inside the notebook**. In Colab, the notebook cannot reach `localhost` directly, so the server's port is shown through Colab's proxy (`serve_kernel_port_as_iframe`); locally, an `IFrame` points to it. Click a node to see its attributes and weights. You can also download the `.onnx` file and drop it on [netron.app](https://netron.app).

In [ ]:
import netron
from IPython.display import IFrame

host, port = netron.start(
    str(onnx_path), browse=False
)  # picks a free port, returns the address
print(f"Netron is serving {onnx_path.name} at http://{host}:{port}")
if IN_COLAB:
    from google.colab import output

    output.serve_kernel_port_as_iframe(port, height=800)
else:
    display(IFrame(f"http://{host}:{port}", width="100%", height=800))

**Finding the U-Net in the Netron graph** (top to bottom):

- **Encoder path:** repeating `Conv` → `Relu` → `Conv` → `Relu` groups (each a `DoubleConv`; the exporter usually folds batch normalization into the convolutions for inference, so `BatchNormalization` nodes may not appear), separated by `MaxPool` nodes, where the size halves.
- **Bottleneck:** the `Conv` group after the fourth `MaxPool`.
- **Decoder path:** `ConvTranspose` nodes (the upsampling), each followed by a **`Concat`** node with **two inputs**. One input comes from the `ConvTranspose`, the other is a long edge from an encoder `Relu` far above: **that edge is a skip connection**. Following the four long edges is the clearest way to see the U.
- **Head:** the last `Conv` (1 x 1 kernel, `W` of shape `[1, 32, 1, 1]`) produces the output `logits` `[1, 1, 256, 256]`.

### The same graph with torchview

`torchview` draws the graph from the PyTorch model directly (no export), with the shape of every tensor. With `depth=2`, each `DoubleConv` and `UpBlock` is one box; the arrows that skip across the drawing are the skip connections, entering `cat` nodes. The SVG is embedded in the notebook (scroll inside the frame).

In [ ]:
from IPython.display import HTML
from torchview import draw_graph

graph = draw_graph(
    UNet(), input_size=(1, 3, 256, 256), depth=2, expand_nested=True, graph_name="UNet"
)
svg = graph.visual_graph.pipe(format="svg").decode()
display(
    HTML(
        f'<div style="height: 900px; overflow: auto; border: 1px solid #ccc">{svg}</div>'
    )
)

## 10. A training loop in plain PyTorch

Before using PyTorch Lightning, here is the whole training procedure written out: the baseline of `config01` (U-Net, BCE, Adam with learning rate 0.001, flips only) for a few epochs. Every epoch:

1. **Train:** for each batch, forward pass → loss → `optimizer.zero_grad()` → `loss.backward()` (gradients) → `optimizer.step()` (update the weights). `model.train()` makes batch normalization use the batch's statistics.
2. **Validate:** with `model.eval()` and `torch.no_grad()` (no gradients, batch normalization uses its running statistics), count over all validation pixels the true positives, false positives, and false negatives, and compute **Dice** and **IoU** from them.

On a GPU it trains 5 epochs (under a minute); on a CPU, 1 epoch of 10 batches, enough to see it work. The real experiments (section 11) train up to 100 epochs.

In [ ]:
import time

L.seed_everything(SEED)
NUM_EPOCHS = 5 if device.type == "cuda" else 1
MAX_BATCHES = None if device.type == "cuda" else 10  # limit the work on a CPU

loop_data = KvasirSegDataModule(
    str(DATA_DIR), image_size=256, batch_size=16, num_workers=2, augmentation="minimal"
)
loop_data.prepare_data()
loop_data.setup("fit")
train_loader, val_loader = loop_data.train_dataloader(), loop_data.val_dataloader()


def evaluate(
    model: torch.nn.Module, loader, threshold: float = 0.5
) -> dict[str, float]:
    """Mean BCE per pixel, and Dice and IoU over all pixels of the loader."""
    model.eval()
    total_loss, pixels, tp, fp, fn = 0.0, 0, 0.0, 0.0, 0.0
    with torch.no_grad():
        for images_b, masks_b in loader:
            images_b, masks_b = images_b.to(device), masks_b.to(device)
            logits_b = model(images_b)
            total_loss += F.binary_cross_entropy_with_logits(
                logits_b, masks_b, reduction="sum"
            ).item()
            pixels += masks_b.numel()
            preds_b = (torch.sigmoid(logits_b) >= threshold).float()
            tp += (preds_b * masks_b).sum().item()
            fp += (preds_b * (1 - masks_b)).sum().item()
            fn += ((1 - preds_b) * masks_b).sum().item()
    return {
        "loss": total_loss / pixels,
        "dice": 2 * tp / (2 * tp + fp + fn),
        "iou": tp / (tp + fp + fn),
    }


model = UNet(base_channels=32, depth=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
for epoch in range(NUM_EPOCHS):
    model.train()
    start, running_loss, seen = time.perf_counter(), 0.0, 0
    for step, (images_b, masks_b) in enumerate(train_loader):
        if MAX_BATCHES is not None and step >= MAX_BATCHES:
            break
        images_b, masks_b = (
            images_b.to(device),
            masks_b.to(device),
        )  # [B, 3, H, W], [B, 1, H, W]
        logits_b = model(images_b)  # [B, 1, H, W]
        loss = F.binary_cross_entropy_with_logits(logits_b, masks_b)  # one number
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch == 0 and step == 0:
            print(
                f"First batch: images {tuple(images_b.shape)}, masks {tuple(masks_b.shape)}, "
                f"logits {tuple(logits_b.shape)}, loss {loss.item():.4f}"
            )
        running_loss += loss.item() * len(images_b)
        seen += len(images_b)
    val = evaluate(model, val_loader)
    history.append(
        {
            "epoch": epoch,
            "train_loss": running_loss / seen,
            **{f"val_{k}": v for k, v in val.items()},
        }
    )
    print(
        f"epoch {epoch}: train loss {running_loss / seen:.4f} | val loss {val['loss']:.4f}, "
        f"Dice {val['dice']:.4f}, IoU {val['iou']:.4f} | {time.perf_counter() - start:.1f} s"
    )

In [ ]:
epochs_axis = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), layout="constrained")
axes[0].plot(epochs_axis, [h["train_loss"] for h in history], "o-", label="train")
axes[0].plot(epochs_axis, [h["val_loss"] for h in history], "o--", label="validation")
axes[0].set_title("BCE loss")
axes[1].plot(epochs_axis, [h["val_dice"] for h in history], "o-", label="Dice")
axes[1].plot(epochs_axis, [h["val_iou"] for h in history], "o-", label="IoU")
axes[1].set_title("validation overlap")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
plt.show()

model.eval()
val_images, val_masks = next(iter(val_loader))
with torch.no_grad():
    val_probs = torch.sigmoid(model(val_images[:4].to(device))).cpu()
fig, axes = plt.subplots(4, 4, figsize=(13, 13), layout="constrained")
for row, image, mask, prob in zip(axes, val_images, val_masks, val_probs):
    row[0].imshow(denormalize(image))
    row[1].imshow(mask[0], cmap="gray")
    row[2].imshow(prob[0], cmap="magma", vmin=0, vmax=1)
    row[3].imshow(prob[0] >= 0.5, cmap="gray")
for ax, title in zip(
    axes[0], ["image", "ground truth", "probability map", "prediction (>= 0.5)"]
):
    ax.set_title(title)
for ax in axes.flat:
    ax.axis("off")
plt.show()

After a few epochs the network already finds some polyps; the probability map shows where it hesitates (orange, around 0.5). The validation Dice jumps up and down in these first epochs: the weights change a lot from one epoch to the next, batch normalization's running statistics (used in eval mode) are still settling, and the 0.5 threshold flips many pixels at once. The full runs of section 11 train long enough for it to settle.

### From this loop to PyTorch Lightning

The project trains the same way, but splits the loop into named pieces and lets Lightning's `Trainer` run them:

| In the loop above | In the project |
| --- | --- |
| `UNet(...)` | `models/components/unet.py` (and `resnet_unet.py`), built by `build_net` from the config's `"model"` section |
| Loss and Dice/IoU counting | `LitUNet._shared_step` (`models/lit_unet.py`), with torchmetrics: `BinaryF1Score` (= Dice), `BinaryJaccardIndex` (= IoU), precision, recall, pixel accuracy |
| `torch.optim.Adam(...)` | `LitUNet.configure_optimizers`: Adam or AdamW, two learning rates, optional cosine schedule |
| `for epoch ...`, `model.train()`, `zero_grad`, `backward`, `step`, `model.eval()`, `torch.no_grad()`, `.to(device)` | `L.Trainer(...).fit(model, datamodule=data)`, in `main.py` |
| `print(...)` per epoch | CSV logger (`metrics.csv`) and W&B logger |
| (nothing) | Best and last checkpoints, early stopping, resuming, mixed precision, a W&B gallery of predicted masks after every epoch |

## 11. Three experiments with the project

The three configs are JSON files in `UnetSemanticSegmentationExample/configs/`. They have **the same keys**, so a change is always just a value. The cell below prints every setting and marks the ones that differ between the configs.

In [ ]:
import json

CONFIGS = ["config01", "config02", "config03"]
CONFIG_DIR = REPO / "UnetSemanticSegmentationExample" / "configs"
configs = {c: json.loads((CONFIG_DIR / f"{c}.json").read_text()) for c in CONFIGS}

print(f"{'setting':<32}" + "".join(f"{c:>16}" for c in CONFIGS))
for section in ("data", "model", "training"):
    for key in configs["config01"][section]:
        values = [configs[c][section][key] for c in CONFIGS]
        changed = "  <- differs" if len({json.dumps(v) for v in values}) > 1 else ""
        print(
            f"{section + '.' + key:<32}"
            + "".join(f"{json.dumps(v):>16}" for v in values)
            + changed
        )

**What each setting does** (copy a config, e.g. to `config04.json`, change values, and run it):

| Setting | Values | Effect |
| --- | --- | --- |
| `data.image_size` | 256 (must be divisible by 32) | Side of the square images. Larger keeps small polyps visible but is slower |
| `data.batch_size` | 16 | Images per training step |
| `data.augmentation` | `"none"`, `"minimal"`, `"strong"` | Training transforms (section 5) |
| `data.split_sizes` | `[800, 100, 100]` | Train, validation, test images (at most 1,000 in total) |
| `model.architecture` | `"unet"`, `"resnet34_unet"` | U-Net from scratch, or with the pretrained ResNet-34 encoder |
| `model.base_channels`, `model.depth` | 32, 4 | U-Net width and number of poolings (`"unet"` only) |
| `model.pretrained` | `true`, `false` | ImageNet weights for the ResNet-34 encoder |
| `training.epochs` | 100 | Maximum number of epochs |
| `training.loss` | `"bce"`, `"dice"`, `"bce_dice"`, `"focal"` | Loss function (section 6; focal loss down-weights easy pixels) |
| `training.optimizer` | `"adam"`, `"adamw"` | AdamW applies weight decay correctly (decoupled from the gradient) |
| `training.lr`, `training.encoder_lr` | 0.001, `null` or 0.0001 | Learning rate; the encoder's own learning rate when set |
| `training.weight_decay` | 0, 0.0001 | Shrinks the weights a little each step, against overfitting |
| `training.scheduler` | `"none"`, `"cosine"` | Cosine: the learning rate decreases smoothly to 0 at the last epoch |
| `training.freeze_encoder_epochs` | 0, 3 | Epochs during which the encoder is frozen (`"resnet34_unet"` only) |
| `training.threshold` | 0.5 | Probability above which a pixel counts as polyp in the metrics |
| `training.precision` | `"16-mixed"`, `"32-true"` | Mixed precision on a GPU: faster, less memory |
| `training.early_stopping`, `training.patience` | `true`, 15 | Stop when validation Dice has not improved for 15 epochs |

**Both options below run exactly the same training** and save results to the same folder (Google Drive in Colab, `runs/` locally). Each run trains, keeps the checkpoint with the best validation Dice, evaluates it on the validation and test sets, and logs everything to W&B. Each takes about 10 to 20 minutes on Colab's T4 GPU (most of it decoding and resizing the JPEG images; config03 usually stops early), a few minutes on a recent GPU.

### Option A: Run as a terminal command

**A1. From a notebook cell (everyone):** a line starting with `!` is sent to the shell, and `{REPO}` is replaced by the repo folder. Locally, if it reports a missing package, the shell is not using the `.venv`'s Python: use Option B, or run the same command in your own terminal with the `.venv` activated.

In [ ]:
!cd "{REPO}" && for c in config01 config02 config03; do python -m UnetSemanticSegmentationExample --config $c.json; done

**A2. From Colab's terminal (Colab Pro):** open the terminal from the **Terminal** button at the bottom left of the Colab window, then run:

```bash
cd /content/CodeCS4337Fall2026
python -m UnetSemanticSegmentationExample --config config01.json
python -m UnetSemanticSegmentationExample --config config02.json
python -m UnetSemanticSegmentationExample --config config03.json
```

The terminal runs on the same machine as this notebook, so it uses the same files, packages, and Drive mount. **You still need to run the setup cells above first**: Drive can only be mounted from a notebook cell, and Colab Secrets can only be read from a notebook cell (the API-key cell saves them to `.env`, where the terminal finds them).

### Option B: Run from Python

Each call trains one config and returns its test metrics; run a single config by keeping only its name in the list.

In [ ]:
from UnetSemanticSegmentationExample import main

test_metrics = {}
for config in ["config01.json", "config02.json", "config03.json"]:
    test_metrics[config] = main(config)

### What gets saved, and what W&B shows

Each run is saved under `<OUTPUT_DIR>/UnetSemanticSegmentationExample/<config>/<timestamp>/`: `config.json`, `metrics.csv` (every metric of every epoch), `hparams.json`, `best_epochNN_valdiceX.XXXX.ckpt`, `last.ckpt`, and W&B's files. Each config folder has a `runs_summary.csv` with one row per run. Sections 13 to 16 read these files.

In the W&B project `UnetSemanticSegmentationExample` (the link is printed at the end of each run), the runs are grouped by config:

- **Curves:** `train_loss`, `val_loss`, `val_dice`, `val_iou`, `val_precision`, `val_recall`, `val_pixel_acc` per epoch; `lr-Adam/...` or `lr-AdamW/pg1`, `pg2` (the learning rate of each parameter group: for config03, `pg1` is the encoder and `pg2` the decoder); `encoder_frozen` (1 while the encoder is frozen).
- **`val_progress`:** the same 8 validation images after every epoch, with the ground truth and the prediction as switchable mask layers. Drag the step slider to watch the masks improve.
- **After training, for the best checkpoint:** `val_per_image` and `test_per_image` tables (Dice, IoU, polyp area, and predicted area of every image; sort by `dice` to find the hardest ones), and `val_worst_examples` / `test_worst_examples` and `..._best_examples` galleries.
- **Summary:** `total_params`, `trainable_params`, `best_epoch`, `train_minutes`, `model_size_mb`, `inference_ms_per_image`.

## 12. Inside the fine-tuned model

`config03` replaces the U-Net's encoder by **ResNet-34**, a classification network already trained on ImageNet (1.2 million photos, 1,000 classes). Its first layers detect edges, colors, and textures, which are useful for polyps too: the network does not have to learn them from 800 images. This is **transfer learning**.

`ResNetUNet` (`models/components/resnet_unet.py`) keeps ResNet-34's five stages as the encoder (without its classifier) and adds a **new decoder** of the same kind as the U-Net's: upsample, concatenate the encoder stage of the same size, convolve.

```text
encoder.stem    ResNet conv 7x7 /2 + BN + ReLU   [ 64, 128, 128] ── skip ─┐            PRETRAINED
encoder.layer1  max-pool + 3 residual blocks      [ 64,  64,  64] ── skip ─┼─┐          (ImageNet)
encoder.layer2  4 residual blocks                 [128,  32,  32] ── skip ─┼─┼─┐
encoder.layer3  6 residual blocks                 [256,  16,  16] ── skip ─┼─┼─┼─┐
encoder.layer4  3 residual blocks (bottleneck)    [512,   8,   8]          │ │ │ │
decoder[0..4]   up + concat + DoubleConv          [256,16,16] ... [16, 256, 256]           NEW (random)
head            1x1 conv                          [  1, 256, 256] logits                   NEW (random)
```

The cell below builds it (the first time, it downloads the ImageNet weights, 83 MB) and counts which parameters are pretrained and which are new.

In [ ]:
from UnetSemanticSegmentationExample.models import ResNetUNet


def count(params) -> int:
    return sum(p.numel() for p in params)


finetune = ResNetUNet(pretrained=True)
print(f"{'part':<16}{'parameters':>14}   origin")
for name, part in finetune.named_children():
    origin = (
        "pretrained on ImageNet" if name == "encoder" else "new, random initialization"
    )
    print(f"{name:<16}{count(part.parameters()):>14,}   {origin}")
for name, stage in finetune.encoder.items():
    print(f"  encoder.{name:<8}{count(stage.parameters()):>12,}")
print(f"{'total':<16}{count(finetune.parameters()):>14,}")
print(
    f"\nFor comparison, the U-Net of config01/02: {count(UNet().parameters()):,} parameters, all random"
)

### Freezing the encoder

During the first `freeze_encoder_epochs` epochs (3 in config03), the encoder is **frozen**: its parameters get `requires_grad = False`, so the optimizer does not change them. Why? The new decoder starts random; its first, large and noisy gradients would flow back into the pretrained encoder and damage the features worth keeping. Once the decoder makes sense of the encoder's features, the whole network is trained (**fine-tuning**), the encoder with a 10 times smaller learning rate (`encoder_lr` 0.0001 vs. `lr` 0.001).

A frozen encoder also keeps its **batch normalization statistics**: `ResNetUNet.train()` keeps the encoder in eval mode while it is frozen; otherwise every batch would still overwrite the running means and variances learned on ImageNet.

In [ ]:
finetune.set_encoder_frozen(True)
finetune.train()
trainable = count(p for p in finetune.parameters() if p.requires_grad)
frozen = count(p for p in finetune.parameters() if not p.requires_grad)
print(
    f"Frozen encoder: trainable {trainable:,}, frozen {frozen:,} ({frozen / (trainable + frozen):.0%} of the network)"
)
print(
    f"  encoder BatchNorm in training mode? {finetune.encoder['stem'][1].training}; decoder in training mode? {finetune.decoder.training}"
)

finetune.set_encoder_frozen(False)
finetune.train()
print(
    f"Unfrozen: trainable {count(p for p in finetune.parameters() if p.requires_grad):,}"
)
print(f"  encoder BatchNorm in training mode? {finetune.encoder['stem'][1].training}")

The torchinfo summary of the frozen network: the **Trainable** column is `False` for every encoder row, and the totals at the bottom split the parameters into trainable and non-trainable.

In [ ]:
finetune.set_encoder_frozen(True)
display(
    summary(
        finetune,
        input_size=(1, 3, 256, 256),
        depth=2,
        col_names=("output_size", "num_params", "trainable"),
        row_settings=("var_names",),
    )
)
finetune.set_encoder_frozen(False)

### What "pretrained" looks like

The 64 filters of ResNet-34's first convolution (7 x 7 pixels, 3 colors), pretrained on ImageNet, next to the same layer with random weights. The pretrained filters are edge, stripe, and color-blob detectors: exactly the low-level features a network trained from scratch must first learn from the 800 Kvasir-SEG images.

In [ ]:
def filter_grid(weight: torch.Tensor) -> np.ndarray:
    """(64, 3, 7, 7) filters to an 8 x 8 grid image, each filter scaled to 0-1."""
    w = weight.detach().cpu()
    w = (w - w.amin(dim=(1, 2, 3), keepdim=True)) / (
        w.amax(dim=(1, 2, 3), keepdim=True) - w.amin(dim=(1, 2, 3), keepdim=True)
    )
    w = torch.nn.functional.pad(w, (1, 1, 1, 1), value=1.0)
    return w.view(8, 8, 3, 9, 9).permute(0, 3, 1, 4, 2).reshape(72, 72, 3).numpy()


random_net = ResNetUNet(pretrained=False)
fig, axes = plt.subplots(1, 2, figsize=(10, 5), layout="constrained")
axes[0].imshow(filter_grid(finetune.encoder["stem"][0].weight))
axes[0].set_title("encoder.stem conv: pretrained on ImageNet")
axes[1].imshow(filter_grid(random_net.encoder["stem"][0].weight))
axes[1].set_title("the same layer, random initialization")
for ax in axes:
    ax.axis("off")
plt.show()

**Scratch vs. fine-tuned, before training:**

| | U-Net (config01, config02) | ResNet-U-Net (config03) |
| --- | --- | --- |
| Parameters | 7.8 million, all random | 24.4 million: 21.3 million pretrained (encoder) + 3.2 million random (decoder and head) |
| Encoder | 4 `DoubleConv` levels, 32 to 256 channels | ResNet-34: 16 residual blocks, 64 to 512 channels, deeper (5 downsamplings) |
| Upsampling | Learned transposed convolutions | Bilinear interpolation (no weights) |
| Trained in epochs 0-2 | Everything | Decoder and head only (3.2 million) |
| Trained from epoch 3 | Everything | Everything, encoder at learning rate 0.0001 |

Note that config03 changes **two things at once** compared with config02: the encoder architecture (bigger and deeper) and the pretraining. Section 18 suggests the experiment that separates them.

## 13. Learning curves

The cells from here on read the **newest run of each config** from `<OUTPUT_DIR>/UnetSemanticSegmentationExample/`. Run the three configs first (section 11). In Colab, Drive must be mounted (section 1). A config with no run yet is skipped.

`metrics.csv` has one row per logged step: training metrics and validation metrics of the same epoch are in separate rows, so the helper below merges them by epoch.

In [ ]:
import csv

PROJECT_RUNS = OUTPUT_DIR / "UnetSemanticSegmentationExample"


def newest_run(config: str) -> tuple[Path, dict] | None:
    """Run folder and runs_summary.csv row of the newest finished run of a config."""
    summary_csv = PROJECT_RUNS / config / "runs_summary.csv"
    if not summary_csv.exists():
        return None
    with summary_csv.open() as f:
        row = list(csv.DictReader(f))[-1]
    return PROJECT_RUNS / config / row["run_id"], row


def epoch_metrics(run_dir: Path, epochs_run: int) -> dict[str, dict[int, float]]:
    """metrics.csv as {metric: {epoch: value}}, skipping empty cells.

    The rows of the final validation and test of the best checkpoint carry the
    epoch number after the last training epoch; they are left out.
    """
    metrics: dict[str, dict[int, float]] = {}
    learning_rates: dict[str, list[float]] = {}
    with (run_dir / "metrics.csv").open() as f:
        for row in csv.DictReader(f):
            if not row["epoch"]:
                # Learning-rate rows (logged at the start of every epoch) have no epoch.
                for key, value in row.items():
                    if key.startswith("lr-") and value:
                        learning_rates.setdefault(key, []).append(float(value))
                continue
            if int(row["epoch"]) >= epochs_run:
                continue
            for key, value in row.items():
                if value and key not in ("epoch", "step"):
                    metrics.setdefault(key, {})[int(row["epoch"])] = float(value)
    epochs = sorted(metrics.get("train_loss", {}))
    for key, values in learning_rates.items():
        metrics[key] = dict(zip(epochs, values))
    return metrics


runs = {c: newest_run(c) for c in CONFIGS}
runs = {c: r for c, r in runs.items() if r is not None}
curves = {
    c: epoch_metrics(run_dir, int(row["epochs_run"]))
    for c, (run_dir, row) in runs.items()
}
for c, (run_dir, row) in runs.items():
    print(
        f"{c}: {run_dir.name}, {row['epochs_run']} epochs, best epoch {row['best_epoch']}"
    )
if not runs:
    print(f"No runs in {PROJECT_RUNS} yet: train the configs in section 11 first")

In [ ]:
COLORS = {"config01": "tab:blue", "config02": "tab:orange", "config03": "tab:green"}
panels = [
    ("loss", "loss (config01: BCE; 02, 03: BCE + Dice)"),
    ("dice", "Dice"),
    ("iou", "IoU"),
    ("precision", "precision"),
    ("recall", "recall"),
    ("pixel_acc", "pixel accuracy"),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 8), layout="constrained")
for ax, (metric, title) in zip(axes.flat, panels):
    for c, m in curves.items():
        for split, style in (("train", "-"), ("val", "--")):
            values = m.get(f"{split}_{metric}", {})
            ax.plot(
                list(values),
                list(values.values()),
                style,
                color=COLORS[c],
                label=f"{c} {split}",
            )
        best = int(runs[c][1]["best_epoch"])
        if best in m.get(f"val_{metric}", {}):
            ax.plot(best, m[f"val_{metric}"][best], "*", color=COLORS[c], markersize=12)
    ax.set_title(title)
    ax.set_xlabel("epoch")
ax = axes.flat[6]
for c, m in curves.items():
    lr_keys = sorted(
        k for k in m if k.startswith("lr-")
    )  # "lr-Adam", or "lr-AdamW/pg1" and "/pg2"
    for key in lr_keys:
        group = (
            {"pg1": "encoder", "pg2": "decoder"}.get(key.rsplit("/", 1)[-1])
            if len(lr_keys) > 1
            else "network"
        )
        ax.plot(
            list(m[key]),
            list(m[key].values()),
            color=COLORS[c],
            linestyle=":" if group == "encoder" else "-",
            label=f"{c} {group}",
        )
ax.set_yscale("log")
ax.set_title("learning rate (log scale)")
ax.set_xlabel("epoch")
ax.legend(fontsize=8)
axes.flat[0].legend(fontsize=8, ncols=2)
axes.flat[7].axis("off")
axes.flat[7].text(
    0,
    0.5,
    "solid: training\ndashed: validation\nstar: best epoch (highest val Dice)\n\n"
    "Training metrics are measured on\naugmented images while the weights\nchange during the epoch.",
    fontsize=11,
    va="center",
)
plt.show()

**Reading the curves:**

- **Loss values are not comparable across configs** when the losses differ: config01 reports BCE, config02 and config03 BCE + Dice (a larger number for the same quality). Compare each config's train and validation loss with each other, and compare configs with Dice and IoU.
- **Gap between training and validation:** a training Dice far above the validation Dice is overfitting. Strong augmentation (config02, config03) makes the training images harder, so their training curves stay closer to, or even below, validation.
- **Precision vs. recall:** BCE alone tends to be cautious (high precision, low recall: it misses polyp edges and small polyps). The Dice term pushes recall up.
- **config03** starts much higher: after one epoch with a frozen encoder it is already about where the scratch U-Nets are after 20 to 40 epochs, thanks to the pretrained features, and it reaches its best score in a fraction of their epochs. Look at what happens at epoch 3, when the encoder is unfrozen.
- **Learning rate:** flat for config01, a half cosine for config02 and config03; for config03 the encoder's (dotted) is 10 times lower than the decoder's.

## 14. Predictions side by side

The best checkpoint of each config, loaded with `load_model` (section 16 explains it), predicts the **same test images**. Each prediction is colored by its kind of pixel:

- **green**: true positive, polyp found;
- **red**: false positive, background predicted as polyp;
- **yellow**: false negative, polyp missed.

The title gives that image's Dice. The images are picked from small to large polyps.

In [ ]:
from UnetSemanticSegmentationExample import load_model

models = {c: load_model(c, device=str(device)) for c in runs}
test_data = KvasirSegDataModule(
    str(DATA_DIR), image_size=256, batch_size=16, num_workers=2
)
test_data.setup("test")
test_set = test_data.test_set

areas = torch.tensor([test_set[i][1].mean().item() for i in range(len(test_set))])
picked = areas.argsort()[torch.linspace(5, len(test_set) - 1, 6).long()].tolist()
samples = torch.stack([test_set[i][0] for i in picked])
truths = torch.stack([test_set[i][1] for i in picked])
probabilities = {}
with torch.no_grad():
    for c, m in models.items():
        probabilities[c] = torch.sigmoid(m(samples.to(device))).cpu()


def error_map(image: np.ndarray, truth: np.ndarray, pred: np.ndarray) -> np.ndarray:
    """Color true positives green, false positives red, false negatives yellow."""
    out = overlay(image, truth * pred, color=(0, 255, 0), alpha=0.5)
    out = overlay(out, (1 - truth) * pred, color=(255, 0, 0), alpha=0.6)
    return overlay(out, truth * (1 - pred), color=(255, 255, 0), alpha=0.6)


def dice_of(truth: np.ndarray, pred: np.ndarray) -> float:
    total = truth.sum() + pred.sum()
    return 1.0 if total == 0 else 2 * (truth * pred).sum() / total


columns = ["image", "ground truth"] + [f"{c}" for c in models]
fig, axes = plt.subplots(
    len(picked),
    len(columns),
    figsize=(3.3 * len(columns), 3.3 * len(picked)),
    layout="constrained",
)
for r, i in enumerate(picked):
    image = denormalize(samples[r])
    truth = truths[r, 0].numpy().astype(np.uint8)
    axes[r, 0].imshow(image)
    axes[r, 0].set_ylabel(test_set.names[i][:12], fontsize=8)
    axes[r, 1].imshow(overlay(image, truth))
    axes[r, 1].set_title(f"{truth.mean():.1%} polyp", fontsize=9)
    for k, c in enumerate(models, start=2):
        pred = (probabilities[c][r, 0] >= 0.5).numpy().astype(np.uint8)
        axes[r, k].imshow(error_map(image, truth, pred))
        axes[r, k].set_title(f"{c}: Dice {dice_of(truth, pred):.2f}", fontsize=9)
axes[0, 0].set_title("image", fontsize=9)
axes[0, 1].set_title("ground truth\n" + axes[0, 1].get_title(), fontsize=9)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

The **probability maps** of the same images show *how sure* each model is, before the 0.5 threshold. Sharp, saturated maps (black and white) mean a confident model; wide orange zones around the polyp mean uncertain boundaries. A model can have the same thresholded mask as another but very different confidence.

In [ ]:
fig, axes = plt.subplots(
    len(picked),
    len(models),
    figsize=(3.6 * len(models), 3.3 * len(picked)),
    layout="constrained",
    squeeze=False,
)
for r in range(len(picked)):
    for k, c in enumerate(models):
        shown = axes[r, k].imshow(probabilities[c][r, 0], cmap="magma", vmin=0, vmax=1)
        axes[r, k].contour(truths[r, 0], levels=[0.5], colors="cyan", linewidths=0.8)
        axes[r, k].axis("off")
        if r == 0:
            axes[r, k].set_title(f"{c}: probability (cyan: true outline)", fontsize=9)
fig.colorbar(shown, ax=axes, shrink=0.4, label="probability of polyp")
plt.show()

## 15. Numbers side by side

The table below collects, for the newest run of each config, the scores of its **best checkpoint** (the epoch with the highest validation Dice) and what it cost. Everything comes from `runs_summary.csv`.

In [ ]:
from IPython.display import HTML

COLUMNS = [
    ("architecture", "model"),
    ("loss", "loss"),
    ("augmentation", "augmentation"),
    ("best_val_loss", "val loss"),
    ("best_val_dice", "val Dice"),
    ("test_dice", "test Dice"),
    ("test_iou", "test IoU"),
    ("test_precision", "test precision"),
    ("test_recall", "test recall"),
    ("test_pixel_acc", "test pixel acc."),
    ("best_epoch", "best epoch"),
    ("epochs_run", "epochs run"),
    ("train_minutes", "training (min)"),
    ("trainable_params", "trainable params"),
    ("model_size_mb", "size (MB)"),
    ("inference_ms_per_image", "inference (ms/image)"),
]


def fmt(key: str, value: str) -> str:
    return f"{int(value):,}" if key == "trainable_params" else value


header = "".join(f"<th>{title}</th>" for _, title in COLUMNS)
body = "".join(
    f"<tr><th>{c}</th>"
    + "".join(f"<td>{fmt(k, row.get(k, ''))}</td>" for k, _ in COLUMNS)
    + "</tr>"
    for c, (_, row) in runs.items()
)
display(HTML(f"<table><tr><th>config</th>{header}</tr>{body}</table>"))
if runs:
    device_used = next(iter(runs.values()))[1].get("inference_device", "?")
    print(
        f"Inference time: one validation batch on the training device ({device_used}), per image, 32-bit"
    )

**Questions to discuss** (answer them with the curves, the predictions, and this table):

1. **Why can't you compare `val loss` between config01 and the others?** (Hint: which loss does each one compute?) Which metric should you use instead?
2. **What did config02 change, and which change mattered most?** It changed four things at once (augmentation, loss, optimizer, schedule). How would you find out which one helped? (Section 18.)
3. **Precision vs. recall:** does adding the Dice loss raise recall? Find a test image in section 14 where config01 misses part of a polyp (yellow) and config02 does not.
4. **Why does config03 reach its best epoch earlier** (if it does)? What does the pretrained encoder bring that 800 images cannot teach?
5. **Is bigger always slower?** config03 has about 3 times more parameters and a 3 times larger file than the U-Net. Compare the inference times. If config03 is not slower, explain why with section 8: where does the U-Net spend its multiply-adds, and how early does ResNet-34 shrink the image? Would each model run in real time on a colonoscopy video (25 to 30 images per second, i.e. 33 to 40 ms per image)?
6. **Pixel accuracy is above 0.9 for every model.** Why is it nearly useless for comparing them? (Section 4: how much of an image is background?)
7. **Validation vs. test:** the best epoch is chosen on the validation set. Why is the test score the honest one, and why can the two differ with only 100 images each?
8. **What do the worst examples have in common?** Open the `test_worst_examples` gallery in W&B (or sort `test_per_image` by Dice): small polyps? flat ones? reflections? the green box?

## 16. Load a trained model, keep training

### Load a trained model

Running a config again trains a **new** model from scratch. To use a model you already trained (after a runtime reset, to test it again, or to try it on your own images), load it with `load_model`: it finds the checkpoint under `<OUTPUT_DIR>/UnetSemanticSegmentationExample/`, rebuilds the network from that run's `config.json`, and loads all its weights (the ImageNet weights are not downloaded again). Nothing is trained. Section 14 used it.

| Call | What it loads |
| --- | --- |
| `load_model("config01")` | The newest run of `config01`, its best checkpoint (`best_*.ckpt`) |
| `load_model("config01/2026-10-04_12-29-09")` | One specific run: the folder name of the run, under `UnetSemanticSegmentationExample/config01/` |
| `load_model("config01", which="last")` | The last epoch instead of the best one |
| `load_model("config01/2026-10-04_12-29-09/best_epochNN_valdiceX.XXXX.ckpt")` | One specific checkpoint file |

The model comes back in eval mode, on the CPU (add `device="cuda"` for the GPU). It expects images preprocessed like in training (resized to 256 x 256, ImageNet-normalized, see `build_transform`) and returns logits:

```python
model = load_model("config03", device="cuda")
masks = torch.sigmoid(model(images.cuda())) >= 0.5   # images: (N, 3, 256, 256)
```

### Keep training a run

To **continue training** an earlier run, for more epochs or to finish a run cut off by a Colab disconnect, pass `resume_from` to `main`. Training continues from the run's last checkpoint (`last.ckpt`, saved after every epoch) and runs the config's `"epochs"` **more** epochs.

| Call | What it continues |
| --- | --- |
| `main("config02.json", resume_from="config02")` | The newest run of `config02`, for `"epochs"` more epochs |
| `main("config02.json", resume_from="config02/2026-10-04_12-29-09")` | One specific run (its folder name) |
| `main("config02_more.json", resume_from="config02")` | A different number of extra epochs: a copy of the config with another `"epochs"` (saved under `config02_more/`) |

- **From the checkpoint:** the weights, the epoch count, the optimizer state (learning rates, weight decay), and the learning-rate schedule. Changing `lr` in the config has no effect when resuming.
- **From the config:** the number of extra `"epochs"`, the data settings, the seed, and logging. The **model settings must match the run's**: the saved weights only fit the same network.
- **Early stopping** starts counting again, and the run must beat the earlier run's best `val_dice`.
- The continued run is saved as a **new run folder** and gets its own row in `runs_summary.csv` (with `resumed_from`). The earlier run is not changed.

The same from a terminal: `python -m UnetSemanticSegmentationExample --config config02.json --resume-from config02`.

In [ ]:
from UnetSemanticSegmentationExample import main

# The config's "epochs" more epochs, on top of the newest config02 run.
main("config02.json", resume_from="config02")

## 17. Try it yourself: other semantic segmentation datasets

The same pipeline (Dataset → DataLoader → U-Net → loss → metrics) works for any segmentation dataset. Pick one below, write a new `Dataset` that returns `(image, mask)` like `KvasirSegDataset`, and adapt what the last column says. The first four are the small medical datasets suggested for this course; the others cover driving scenes, aerial images, pets, people, and industrial inspection. **Include your choice and your results in this table** when you report your experiments.

| Dataset | Domain | Difficulty | Classes | Why it is interesting | What to change |
| --- | --- | --- | --- | --- | --- |
| [**Kvasir-SEG**](https://datasets.simula.no/kvasir-seg/) (this project) | Medical: colonoscopy | Easy to medium | 2 (polyp, background) | One zip, no login; polyps of every size; a well-known benchmark to compare with published scores | Nothing; try `image_size` 352 or 512 for small polyps |
| [**CHASE_DB1**](https://huggingface.co/datasets/MedOtter/CHASE_DB1) | Medical: retinal blood vessels | Hard | 2 (vessel, background) | Only 28 images, 999 x 960: truly tiny data; vessels 1 to 10 pixels wide; about 90% background | Train on random 128 x 128 or 256 x 256 **crops** (patches), predict the full image by tiles; BCE + Dice; masks are the first observer's annotations; report recall (sensitivity) and Dice; resizing the whole image would erase the thin vessels |
| [**BUSI**](https://zenodo.org/records/21128640) (Breast Ultrasound Images) | Medical: breast ultrasound | Medium | 2 (lesion, background); each image also labeled normal, benign, or malignant | Noisy, low-contrast grayscale images; segmentation and classification from one dataset | Grayscale: repeat the channel 3 times (for the pretrained encoder) or `in_channels=1`; some images have several mask files (`_mask_1.png`), merge them with a logical OR; "normal" images have empty masks (keep them to learn "no lesion", but their Dice is 1 or 0); 256 x 256 |
| [**MSD Task04 Hippocampus**](http://medicaldecathlon.com/) (Medical Segmentation Decathlon) | Medical: brain MRI, 3D | Medium to hard | 3 (background, anterior, posterior hippocampus) | Small 3D volumes (263 labeled, 29 MB); a first multiclass and 3D project | Read NIfTI (`.nii.gz`) files with `nibabel` (a new dependency) and train on 2D slices; **multiclass**: `out_channels=3`, cross-entropy + multiclass Dice, mask as `int64` class indices `[H, W]`; slices are about 35 x 51 pixels, pad to 64 x 64 and use `depth=3`; per-class Dice |
| [**Oxford-IIIT Pet**](https://www.robots.ox.ac.uk/~vgg/data/pets/) | Everyday photos: cats and dogs | Easy | 3 in the trimap (pet, background, border) | Built into torchvision (`OxfordIIITPet(target_types="segmentation")`); 7,349 images; large, centered objects | Trimap values are 1, 2, 3: subtract 1; either binary (pet vs. rest) or 3 classes with `ignore_index` on the uncertain border; 256 x 256 |
| [**CamVid**](https://www.kaggle.com/datasets/carlolepelaars/camvid) | Driving: city streets | Medium to hard | 32 (usually merged into 11 + void) | Many classes of very different sizes (road vs. poles and signs); small (701 images) | Masks are **RGB colors**: map each color to a class index; `out_channels=11` or 12; cross-entropy with class weights and `ignore_index` for void; images are 360 x 480, and 360 is not divisible by 16 (`depth=4`): crop or pad to 352 x 480 or 384 x 480; mean IoU (`MulticlassJaccardIndex`) |
| [**Cityscapes**](https://www.cityscapes-dataset.com/) | Driving: 50 European cities | Hard | 19 evaluated (30 annotated) | The standard driving benchmark; 5,000 finely annotated 2048 x 1024 images; free registration required | Convert label IDs to the 19 train IDs (255 = ignore); random 512 x 512 crops; `out_channels=19`; cross-entropy with `ignore_index=255`; mean IoU; a pretrained encoder is almost mandatory |
| [**Massachusetts Buildings**](https://www.kaggle.com/datasets/balraj98/massachusetts-buildings-dataset) | Aerial / satellite: buildings | Medium | 2 (building, background) | 151 tiles of 1500 x 1500 at 1 m per pixel; many small objects per image | Cut tiles into 256 x 256 patches (do not resize: buildings would vanish); BCE + Dice; IoU; for [Inria Aerial Image Labeling](https://project.inria.fr/aerialimagelabeling/), the same with 5000 x 5000 tiles |
| [**Supervisely Persons**](https://www.kaggle.com/datasets/tapakah68/supervisely-filtered-segmentation-person-dataset) | People: portraits and scenes | Medium | 2 (person, background) | Thin structures (hair, fingers) and many poses; the basis of "background blur" in video calls | Masks may be stored as color images: binarize; 320 x 320 or larger; boundary quality matters, so look at IoU and the predictions, not pixel accuracy |
| [**Magnetic Tile Defects**](https://github.com/abin24/Magnetic-tile-defect-datasets.) | Industrial inspection | Medium to hard | 6 folders (5 defect types + free); binary mask per image | Tiny defects (blowholes, cracks, breaks, frays, uneven surfaces), extreme imbalance; grayscale | Binary (defect vs. not) or 6 classes from the folder name; focal or Dice loss; grayscale input; recall matters most (a missed crack is worse than a false alarm) |
| [**MVTec AD**](https://www.mvtec.com/company/research/datasets/mvtec-ad) | Industrial inspection, 15 objects | Hard | 2 (defect, normal) per object | The training set has **only defect-free images**; masks exist only in the test set: shows the limits of supervised segmentation | Supervised training is only possible on a split of the test set; better seen as anomaly detection; non-commercial license |
| [**ISIC 2018 Task 1**](https://challenge.isic-archive.com/data/#2018) | Medical: skin lesions (dermoscopy) | Easy to medium | 2 (lesion, background) | 2,594 training images, large lesions, hair and rulers as distractors | Images up to 6000 x 4000: resize to 256 or 384; the challenge metric is a thresholded Jaccard (IoU below 0.65 counts as 0) |
| [**DRIVE**](https://drive.grand-challenge.org/) | Medical: retinal vessels | Hard | 2 | The classic vessel benchmark (40 images), like CHASE_DB1 | Patches as for CHASE_DB1; evaluate only inside the provided field-of-view mask |
| [**Montgomery / Shenzhen chest X-rays**](https://lhncbc.nlm.nih.gov/LHC-downloads/downloads.html#tuberculosis-image-data-sets) | Medical: lungs in chest X-rays | Easy | 2 (lung, background) | Large, well-defined organs; quick high Dice | Left and right lung masks are separate files: merge them; grayscale; 256 x 256 |
| [**Carvana**](https://www.kaggle.com/c/carvana-image-masking-challenge) | Products: cars on a turntable | Easy | 2 (car, background) | Very high Dice (above 0.99) is reachable: shows what a nearly solved problem looks like | Kaggle login; 1918 x 1280 images: resize; masks are GIFs |
| [**PASCAL VOC 2012**](http://host.robots.ox.ac.uk/pascal/VOC/voc2012/) | Everyday photos, 20 object classes | Hard | 21 (20 + background) | Built into torchvision (`VOCSegmentation`); the classic multiclass benchmark | Palette PNG masks: values are class indices, 255 = border (ignore); `out_channels=21`; cross-entropy with `ignore_index=255`; mean IoU; pretrained encoder |

### Going multiclass

Binary segmentation uses one output channel and BCE. For $C > 2$ classes, the changes are always the same:

| | Binary (this project) | Multiclass |
| --- | --- | --- |
| Network output | `UNet(out_channels=1)` → `[B, 1, H, W]` | `UNet(out_channels=C)` → `[B, C, H, W]` |
| Mask encoding | `float32` 0.0 / 1.0, `[B, 1, H, W]` | `int64` class index per pixel, `[B, H, W]` (convert colors or palette values to indices) |
| Loss | `F.binary_cross_entropy_with_logits` (+ Dice) | `F.cross_entropy(logits, mask, ignore_index=255)` (+ multiclass Dice) |
| Prediction | `sigmoid(logits) >= 0.5` | `logits.argmax(dim=1)` |
| Metrics | `BinaryF1Score` (Dice), `BinaryJaccardIndex` (IoU) | `MulticlassJaccardIndex(num_classes=C, average="macro")` (mean IoU), per-class IoU with `average="none"` |
| Image size | divisible by `2**depth` (16 here) or 32 for the ResNet encoder | the same |

Resizing must use nearest-neighbor interpolation for masks (`tv_tensors.Mask` does this automatically), so that no "class 2.5" appears between class 2 and class 3.

In [ ]:
# The multiclass changes in a few lines, with random tensors (C = 3 classes, e.g. the pet trimap).
C = 3
multi_net = UNet(out_channels=C)
x = torch.randn(2, 3, 128, 128)
target = torch.randint(0, C, (2, 128, 128))  # int64 class index per pixel
multi_logits = multi_net(x)  # [B, C, H, W]
multi_loss = F.cross_entropy(
    multi_logits, target
)  # softmax over the C channels of each pixel
multi_pred = multi_logits.argmax(dim=1)  # [B, H, W], one class per pixel
print(
    f"logits {tuple(multi_logits.shape)}, target {tuple(target.shape)} {target.dtype}, "
    f"loss {multi_loss.item():.3f} (about ln {C} = {np.log(C):.3f} when untrained), prediction {tuple(multi_pred.shape)}"
)

## 18. Experiment ideas

Each idea is a **hypothesis** you can test by copying a config (e.g. `configs/config04.json`), changing one or two values, running it, and comparing with sections 13 to 15. Change **one thing at a time**, keep the seed, and write down what you expected *before* running.

| # | Experiment | How | Hypothesis | What to look at |
| --- | --- | --- | --- | --- |
| 1 | Which part of config02 helped? | Start from config01 and change only `augmentation`, then only `loss`, then only `scheduler` | Augmentation reduces overfitting most; the Dice loss raises recall | Train vs. validation Dice gap; recall |
| 2 | Pretraining vs. architecture | config03 with `"pretrained": false` (and `freeze_encoder_epochs` 0) | Most of config03's gain comes from the ImageNet weights, not from the bigger encoder | Test Dice of config02 vs. this run vs. config03 |
| 3 | Less data | `"split_sizes": [100, 100, 100]` in config01 and config03 | Pretraining helps much more when there are few training images | Gap between scratch and fine-tuned grows |
| 4 | Higher resolution | `"image_size": 352` or 384 (divisible by 32) | Small polyps are segmented better; training is about 2 times slower | Dice of the images with the smallest polyps (`test_per_image` table, sort by `polyp_area`) |
| 5 | Loss functions | `"loss"`: `"bce"`, `"dice"`, `"bce_dice"`, `"focal"` | Dice alone gives higher recall but noisier masks; BCE alone gives higher precision | Precision vs. recall; probability maps |
| 6 | Decision threshold | Compute Dice on the validation probabilities for thresholds 0.3 to 0.7 (the probabilities of section 14 are a start) | A threshold below 0.5 helps the BCE model, which is too cautious | Validation Dice per threshold; then apply the best one to the test set once |
| 7 | U-Net width | `"base_channels"`: 16, 32, 64 | 16 (2 million parameters) is almost as good; 64 (31 million, the original U-Net) overfits without strong augmentation | Test Dice vs. trainable parameters and inference time |
| 8 | U-Net depth | `"depth"`: 3 or 5 | Depth 3 sees too little context for large polyps; depth 5 brings little | Dice on large vs. small polyps |
| 9 | Freezing schedule | `"freeze_encoder_epochs"`: 0, 3, 100 (always frozen) | Always frozen is worse (ImageNet features are not colonoscopy features); 0 is slightly worse than 3 | Validation Dice in the first epochs; final test Dice |
| 10 | Encoder learning rate | `"encoder_lr"`: 0.001 (same as decoder), 0.0001, 0.00001 | Too high destroys the pretrained features; too low barely adapts them | Validation curves after the unfreezing epoch |
| 11 | No skip connections | In a copy of `UNet.forward`, pass `torch.zeros_like(skip)` instead of `skip` | Masks become blurry and boundaries worse; Dice drops clearly | Predictions at polyp boundaries |
| 12 | Batch size and learning rate | `"batch_size"`: 4, 8, 32 with matching `"lr"` | Small batches make batch normalization noisy; doubling the batch allows a higher learning rate | Curves' smoothness; final Dice |
| 13 | Upsampling type | Replace `ConvTranspose2d` in `UpBlock` by `nn.Upsample(scale_factor=2, mode="bilinear")` + a 1x1 convolution | Bilinear upsampling avoids checkerboard artifacts in the probability maps, with fewer parameters | Probability maps; parameters |
| 14 | Another dataset | Section 17 | The same U-Net works, but thin structures (vessels) or many classes (CamVid) need the changes listed there | Per-class IoU, predictions |

## Pro tips: keep training running

Colab disconnects runtimes that look idle (no interaction for a while, often around 90 minutes) and also caps how long a runtime can run in total (around 12 hours on the free tier, longer on paid plans). The exact limits vary and are not guaranteed. When the runtime disconnects, training stops and everything under `/content/` (outside Drive) is deleted, including the downloaded dataset (it is downloaded again, 44 MB, in seconds).

**Save your state in Google Drive**
- Always mount Drive *before* training. Results then go straight to Drive (`OUTPUT_DIR` in `.envcolab`), not to the temporary disk.
- Two checkpoints are saved to Drive *during* training: the best model so far (`best_epochXX_valdiceY.ckpt`), every time validation Dice improves, and the last epoch (`last.ckpt`), after every epoch.
- The run's other files (`config.json`, `hparams.json`, and the row in `runs_summary.csv`) are written only when the run finishes. To finish an interrupted run, resume it (section 16).
- Keep your notebook in Drive too: *File → Save a copy in Drive*. Keep copies of configs you edited in Drive, since `/content/CodeCS4337Fall2026` is deleted on reset.

**Avoid idle disconnects**
- Keep the Colab browser tab open, and don't let your computer go to sleep.
- Check on the notebook now and then. Running in the terminal does *not* prevent disconnects: the runtime is shared.
- Don't use scripts or browser extensions that fake activity (auto-clickers); they can get your Colab usage restricted.

**Plan your runs**
- Test a new config with few epochs first (e.g. `"epochs": 2`), then start the full run.
- Each `main(...)` call saves its own results: run the configs one at a time if your session is short.
- When you are done, free the GPU with *Runtime → Disconnect and delete runtime*.